<a href="https://colab.research.google.com/github/netphone/prostateX/blob/main/Densenet_train_with_configs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## pre-request installation

In [ ]:
import keras
import tensorflow as tf
print(keras.__version__)
print(tf.__version__)
import numpy as np
print(np.version.version)
import pickle
import pandas as pd
print(pickle.format_version)
print(pd.__version__)


2.2.5
1.14.0
1.16.5
4.0
0.24.2


In [ ]:
from tensorflow.python.client import device_lib
device_lib.list_local_devices()

[name: "/device:CPU:0"
 device_type: "CPU"
 memory_limit: 268435456
 locality {
 }
 incarnation: 12005041617180958370, name: "/device:XLA_CPU:0"
 device_type: "XLA_CPU"
 memory_limit: 17179869184
 locality {
 }
 incarnation: 6022073438003132275
 physical_device_desc: "device: XLA_CPU device", name: "/device:XLA_GPU:0"
 device_type: "XLA_GPU"
 memory_limit: 17179869184
 locality {
 }
 incarnation: 849560015678161353
 physical_device_desc: "device: XLA_GPU device", name: "/device:GPU:0"
 device_type: "GPU"
 memory_limit: 11330115994
 locality {
   bus_id: 1
   links {
   }
 }
 incarnation: 5602516323088051967
 physical_device_desc: "device: 0, name: Tesla K80, pci bus id: 0000:00:04.0, compute capability: 3.7"]

In [ ]:
!pip install git+https://www.github.com/keras-team/keras-contrib.git

  Cloning https://www.github.com/keras-team/keras-contrib.git to /tmp/pip-req-build-xjejhbq7
  Running command git clone -q https://www.github.com/keras-team/keras-contrib.git /tmp/pip-req-build-xjejhbq7
  Created wheel for keras-contrib: filename=keras_contrib-2.0.8-cp36-none-any.whl size=101065 sha256=411e45f5febcd126d007041bdeaa5a102d1cd0320cd6c5dd2427ff589898f267
  Stored in directory: /tmp/pip-ephem-wheel-cache-5byoqqg3/wheels/11/27/c8/4ed56de7b55f4f61244e2dc6ef3cdbaff2692527a2ce6502ba
Successfully built keras-contrib


In [ ]:
!pip3 install -q pydicom
!pip3 install SimpleITK

     |████████████████████████████████| 7.1MB 2.7MB/s 
     |████████████████████████████████| 42.5MB 23.6MB/s 


### load Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')

Go to this URL in a browser: https://accounts.google.com/o/oauth2/auth?client_id=947318989803-6bn6qk8qdgf4n4g3pfee6491hc0brc4i.apps.googleusercontent.com&redirect_uri=urn%3Aietf%3Awg%3Aoauth%3A2.0%3Aoob&scope=email%20https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fdocs.test%20https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fdrive%20https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fdrive.photos.readonly%20https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fpeopleapi.readonly&response_type=code

Enter your authorization code:
··········
Mounted at /content/gdrive


In [ ]:
import os
os.chdir('gdrive/My Drive/Colab Notebooks/ProstateX_trial')

In [ ]:
from DenseNet.DenseNet3D import *
from DenseNet.losses import categorical_focal_loss


Using TensorFlow backend.


### executing dataset_exam_fun.py to collect resized image info
* save resized images into 'resized' and 'resized_ktrans' folders
* save 'resized_dicom_imgs.pkl' and 'resized_ktrans_imgs.pkl'
* save four files named:  sp_res.pkl, ktrans_res.pkl, resize_finding.pkl, and resize_finding_ktrans.pkl

In [ ]:
!python dataset_exam_fun.py --data-path='./raw data' --dicom-dir='DICOM' --ktrans-dir='ProstateX-Ktrans'

(19, 320, 320) (19, 320, 320) (19, 320, 320)
(19, 320, 320) (19, 320, 320) (19, 320, 320)
(19, 320, 320) (19, 320, 320) (19, 320, 320)
(21, 320, 320) (19, 320, 320) (19, 320, 320)
(19, 320, 320) (19, 320, 320) (19, 320, 320)
(19, 320, 320) (19, 320, 320) (19, 320, 320)
(19, 320, 320) (19, 320, 320) (19, 320, 320)
(19, 320, 320) (19, 320, 320) (19, 320, 320)
(21, 320, 320) (23, 320, 320) (21, 320, 320)
(19, 320, 320) (19, 320, 320) (19, 320, 320)
(21, 320, 320) (19, 320, 320) (19, 320, 320)
(19, 320, 320) (19, 320, 320) (19, 320, 320)
(19, 320, 320) (19, 320, 320) (19, 320, 320)
(19, 320, 320) (19, 320, 320) (23, 320, 320)
(21, 320, 320) (19, 320, 320) (19, 320, 320)
(19, 320, 320) (19, 320, 320) (19, 320, 320)
(21, 320, 320) (21, 320, 320) (21, 320, 320)
(19, 320, 320) (19, 320, 320) (19, 320, 320)
(19, 320, 320) (19, 320, 320) (19, 320, 320)
(21, 320, 320) (21, 320, 320) (21, 320, 320)
(19, 320, 320) (19, 320, 320) (19, 320, 320)
(19, 320, 320) (21, 320, 320) (19, 320, 320)
(19, 320, 

### executing Image_Patch_Generation_fun.py to generate finding and randomly-picked image patches with different sizes (** need repeate again to run the revised Image_Patch_Generation_fun.py to fix some finding unmatched issues)
* save 'patch_coord.pkl'
* save 'patch_coord_ktrans.pkl'

In [ ]:
#!python Image_Patch_Generation_fun.py --working-folder='./data' -p '(7, 96, 96)' '(5, 64, 64)' '(5, 48, 48)' '(3, 42, 42)' --overlap=4
!python Image_Patch_Generation_revWktrans.py --working-folder='./raw data' -p '(7, 96, 96)' '(5, 64, 64)' '(5, 48, 48)' '(3, 42, 42)' --overlap=4

Image_Patch_Generation_revWktrans.py:108: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  clinSig = fid_label[fid_label.ProxID == ProxID][fid_label.pos == pos].ClinSig.item()
ProstateX-0025 4-t2tsetra fid3 (234, 168, -8)  is smaller than 0 and will not be included.
k: 31, start_D: 14, end_D: 21
k: 26, start_D: 14, end_D: 21
k: 26, start_D: 14, end_D: 21
k: 26, start_D: 14, end_D: 21
ProstateX-0025 8-ep2ddifftraDYNDISTCALCBVAL fid3 (236, 168, -7)  is smaller than 0 and will not be included.
k: 31, start_D: 12, end_D: 19
k: 26, start_D: 12, end_D: 19
k: 26, start_D: 12, end_D: 19
k: 26, start_D: 12, end_D: 19
ProstateX-0025 7-ep2ddifftraDYNDISTADC fid3 (236, 168, -7)  is smaller than 0 and will not be included.
k: 31, start_D: 12, end_D: 19
k: 26, start_D: 12, end_D: 19
k: 26, start_D: 12, end_D: 19
k: 26, start_D: 12, end_D: 19
ProstateX-0025 4-t2tsetra fid3 (234, 168, -8)  is smaller than 0 and will not be included.
k: 31, start_D: 12, end_D: 19
k: 26, sta

### Execute save_img_patches.py (or save_img_patchesWktrans.py) to create subfolders associated with patch sizes under img_patches (and img_ktrans_patches) folders
* save associated patch-sized .npy files and corrospoinding patch-sized info in the particular patch-size subfolder.


In [ ]:
####################
import os
print(os.getcwd())
os.chdir('./raw data')
print(os.getcwd())
####################
!python save_img_patchesWktrans.py


/content/gdrive/My Drive/Colab Notebooks/ProstateX_trial
/content/gdrive/My Drive/Colab Notebooks/ProstateX_trial/raw data
ProstateX-0000


## Calculate averages and stds, save standardized data and recheck if the process is completed

In [ ]:
!python std.py
!python std_ktrans.py